In [1]:
print('In the name of Allah')

In the name of Allah


# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [2]:
# YOUR CODE HERE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression

df = pd.read_csv('titanic_data_updated.csv')

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
686,687,no,third,"Panula, Mr. Jaako Arnold",male,14.0,4,1,3101295,39.6875,NaN,S
795,796,no,second,"Otter, Mr. Richard",male,39.0,0,0,28213,13.0000,NaN,S
8,9,yes,third,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
275,276,yes,first,"Andrews, Miss. Kornelia Theodosia",female,63.0,1,0,13502,77.9583,D7,S
122,123,no,second,"Nasser, Mr. Nicholas",male,32.5,1,0,237736,30.0708,NaN,C


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [3]:
# YOUR CODE HERE

df['Family_size'] = df['SibSp'] + df['Parch'] + 1

df['Cabin'] = df['Cabin'].fillna('Mising')

df['Deck'] = df['Cabin'].astype(str).str[0]

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Deck
446,447,yes,second,"Mellinger, Miss. Madeleine Violet",female,13.0,0,1,250644,19.5000,Mising,S,2,M
38,39,no,third,"Vander Planke, Miss. Augusta Maria",female,18.0,2,0,345764,18.0000,Mising,S,3,M
108,109,no,third,"Rekic, Mr. Tido",male,38.0,0,0,349249,7.8958,Mising,S,1,M
289,290,yes,third,"Connolly, Miss. Kate",female,22.0,0,0,370373,7.7500,Mising,Q,1,M
72,73,no,second,"Hood, Mr. Ambrose Jr",male,21.0,0,0,S.O.C. 14879,73.5000,Mising,S,1,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [4]:
# YOUR CODE HERE

X = df.drop('Survived', axis = 1)

y = df['Survived']

print(f'Shape of X : {X.shape}')
print(f'Shape of y : {y.shape}')


Shape of X : (891, 13)
Shape of y : (891,)


---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [5]:
# YOUR CODE HERE

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.2, random_state = 42, stratify = y )

print(f'Shape of X_train : {X_train.shape}')
print(f'Shape of X_test : {X_test.shape}')

Shape of X_train : (712, 13)
Shape of X_test : (179, 13)


---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [6]:
# YOUR CODE HERE

age_mean = X_train['Age'].mean()
age_std = X_train['Age'].std()

X_train['Z_score'] = ( X_train['Age'] - age_mean ) / age_std

musk = ( abs( X_train['Z_score'] <= 3 ) )

X_train = X_train[musk]
y_train = y_train[musk]

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

fare_IQR = fare_Q3 - fare_Q1

minimum = max( 0, fare_Q1 - 1.5 * fare_IQR )
maximum = fare_Q3 + 1.5 * fare_IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [7]:
# pipeline for Numerical Columns

p1 = Pipeline(
    steps = [
        ( 'imputer', SimpleImputer(strategy = 'mean') ),
        ( 'scaler', StandardScaler() )
    ]
)

p1

p2 = Pipeline(
    steps = [
        ( 'imputer', SimpleImputer( strategy = 'median' ) ),
        ( 'scaler', MinMaxScaler() )
    ]
)

p2

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', MinMaxScaler())])

---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [8]:
categories = [['third', 'second', 'first']]

In [9]:
# pipeline for categorical columns

p3 = Pipeline(
    steps = [
        ( 'imputer', SimpleImputer( strategy = 'most_frequent' ) ),
        ( 'encoder', OneHotEncoder( sparse_output = False, drop = 'first', handle_unknown = 'ignore' ) )
    ]
)

p3

p4 = Pipeline(
    steps = [
        ( 'imputer' ,SimpleImputer(strategy='most_frequent') ),
        ( 'encoder', OrdinalEncoder(categories = categories) ),
        ( 'scaler', MinMaxScaler() )
    ]
)

p4

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OrdinalEncoder(categories=[['third', 'second', 'first']])),
                ('scaler', MinMaxScaler())])

In [10]:
preprocessor = ColumnTransformer(
    transformers = [
        ( 'pipeline_01', p1, ['Age'] ),
        ( 'pipeline_02', p2, ['Fare', 'Family_size'] ),
        ( 'pipeline_03', p3, ['Embarked', 'Sex', 'Deck'] ),
        ( 'pipeline_04', p4, ['Pclass'] )
    ]
)

preprocessor

ColumnTransformer(transformers=[('pipeline_01',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_02',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_size']),
                                ('pipeline_03',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_04',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [11]:
# YOUR CODE HERE

le = LabelEncoder()

le.fit(y_train)

print(f'Before : ', np.unique(y_train))

y_train = le.transform( y_train )
y_test = le.transform( y_test )

print(f'After : ', np.unique(y_train))

Before :  ['no' 'yes']
After :  [0 1]


---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [12]:
lr_model = Pipeline(
    steps = [
        ( 'preprocessror', preprocessor ),
        ( 'model', LogisticRegression(class_weight = 'balanced', max_iter = 1000) )
    ]
)

lr_model

Pipeline(steps=[('preprocessror',
                 ColumnTransformer(transformers=[('pipeline_01',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_02',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_size']),
                                                 ('pipeline_03',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(str...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_04',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [13]:
lr_model.fit( X_train, y_train )

Pipeline(steps=[('preprocessror',
                 ColumnTransformer(transformers=[('pipeline_01',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_02',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_size']),
                                                 ('pipeline_03',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(str...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_04',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [14]:
y_pred = lr_model.predict(X_test)

print('First 10 predictions : ')
print(y_pred[:10])

First 10 predictions : 
[0 0 0 0 1 1 1 0 1 0]


---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score( y_test, y_pred )
precision = precision_score( y_test, y_pred )
recall = recall_score( y_test, y_pred )

print(f'Accuracy  : {round(accuracy, 2)}')
print(f'Precision : {round(precision, 2)}')
print(f'Recall    : {round(recall, 2)}')

Accuracy  : 0.77
Precision : 0.68
Recall    : 0.75
